<a href="https://colab.research.google.com/github/MilagrosBrucellaria/preentrega_2_ventas/blob/main/preentrega_2_ventas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pre-Entrega 2 - Integración de Datos de Ventas
## Objetivo

El objetivo de este trabajo es simular un escenario real donde la información de ventas proviene de distintas áreas de la organización y se encuentra almacenada en formatos diferentes.

Para ello se trabajará con:

- Un archivo CSV que contiene las transacciones.
- Un archivo Excel que contiene el catálogo de productos.

Posteriormente se realizará la limpieza de datos, la integración de ambas fuentes mediante un merge utilizando `id_producto` como clave, el cálculo de una nueva variable llamada `total_venta` y finalmente la exportación del resultado a formato Parquet.

# Paso 1: Conexión con Google Drive

Se conecta Google Colab con Google Drive para acceder a los archivos fuente y guardar los resultados generados durante el proceso ETL.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
ruta = "/content/drive/MyDrive/Data Science/Data Science II/PreEntrega2_DSII"

print("Ruta utilizada:")
print(ruta)

Ruta utilizada:
/content/drive/MyDrive/Data Science/Data Science II/PreEntrega2_DSII


# Paso 2: Importación de librerías

Se importan las librerías necesarias para la lectura, transformación y exportación de los datos.

- pandas: manipulación de datos.
- openpyxl: lectura de Excel.
- pyarrow: exportación a formato Parquet.

In [3]:
!pip install openpyxl pyarrow -q

import pandas as pd

# Paso 3: Carga de las fuentes de datos

Se cargan dos datasets provenientes de diferentes sistemas:

1. transacciones.csv
2. productos.xlsx

La primera fuente contiene las ventas realizadas y la segunda contiene el catálogo maestro de productos.

In [4]:
transacciones = pd.read_csv(
    f"{ruta}/transacciones.csv")

productos = pd.read_excel(
    f"{ruta}/productos.xlsx",
    engine="openpyxl")

In [5]:
print("TRANSACCIONES")
display(transacciones.head())

print("PRODUCTOS")
display(productos.head())

TRANSACCIONES


,id_transaccion,id_producto,cantidad,fecha
0,1,101,2.0,2026-01-05
1,2,102,1.0,2026-01-06
2,3,103,3.0,2026-01-07
3,4,104,2.0,2026-01-08
4,5,105,4.0,2026-01-09


PRODUCTOS


,id_producto,nombre_producto,precio_unitario
0,101,Notebook,850000
1,102,Monitor,310000
2,103,Teclado,52000
3,104,Mouse,27000
4,105,Auriculares,78000


# Paso 4: Inspección inicial de los datos

Antes de comenzar la transformación es importante revisar la estructura de los datasets para identificar posibles inconsistencias, tipos de datos incorrectos o valores faltantes.

In [6]:
print("Información de transacciones")
transacciones.info()

print("\n")

print("Información de productos")
productos.info()

Información de transacciones
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_transaccion  10 non-null     int64  
 1   id_producto     10 non-null     int64  
 2   cantidad        9 non-null      float64
 3   fecha           9 non-null      object 
dtypes: float64(1), int64(2), object(1)
memory usage: 452.0+ bytes


Información de productos
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id_producto      5 non-null      int64 
 1   nombre_producto  5 non-null      object
 2   precio_unitario  5 non-null      int64 
dtypes: int64(2), object(1)
memory usage: 252.0+ bytes


# Paso 5: Corrección de tipos de datos

Para garantizar que ambas fuentes puedan integrarse correctamente se normalizan los tipos de datos.

Se transforma:

- id_producto a formato numérico.
- cantidad a formato numérico.
- precio_unitario a formato numérico.
- fecha a formato datetime.

In [7]:
transacciones["id_producto"] = pd.to_numeric(
    transacciones["id_producto"],
    errors="coerce"
)

productos["id_producto"] = pd.to_numeric(
    productos["id_producto"],
    errors="coerce"
)

transacciones["cantidad"] = pd.to_numeric(
    transacciones["cantidad"],
    errors="coerce"
)

productos["precio_unitario"] = pd.to_numeric(
    productos["precio_unitario"],
    errors="coerce"
)

transacciones["fecha"] = pd.to_datetime(
    transacciones["fecha"],
    errors="coerce"
)

# Paso 6: Manejo de valores nulos

Se identifican y corrigen valores faltantes.

La variable cantidad se completa utilizando la mediana debido a que es una variable numérica.

Los registros sin fecha son eliminados porque no permiten identificar correctamente cuándo ocurrió la transacción.

In [8]:
transacciones["cantidad"] = (
    transacciones["cantidad"]
    .fillna(transacciones["cantidad"].median())
)

transacciones = transacciones.dropna(
    subset=["fecha"]
)

In [9]:
print("Valores nulos por columna:")

transacciones.isnull().sum()

Valores nulos por columna:


,0
id_transaccion,0
id_producto,0
cantidad,0
fecha,0


# Paso 7: Integración de las fuentes

Se realiza un merge utilizando la columna `id_producto` como llave de integración.

Este proceso permite incorporar el nombre y el precio del producto a cada transacción registrada.

In [10]:
ventas = transacciones.merge(
    productos,
    on="id_producto",
    how="left"
)

ventas.head()

,id_transaccion,id_producto,cantidad,fecha,nombre_producto,precio_unitario
0,1,101,2.0,2026-01-05,Notebook,850000
1,2,102,1.0,2026-01-06,Monitor,310000
2,3,103,3.0,2026-01-07,Teclado,52000
3,4,104,2.0,2026-01-08,Mouse,27000
4,5,105,4.0,2026-01-09,Auriculares,78000


# Paso 8: Creación de la variable total_venta

Se calcula una nueva variable denominada `total_venta`.

La fórmula utilizada es:

total_venta = cantidad × precio_unitario

Esta métrica representa el importe total generado por cada operación de venta.

In [11]:
ventas["total_venta"] = (
    ventas["cantidad"]
    * ventas["precio_unitario"]
)

ventas.head()

,id_transaccion,id_producto,cantidad,fecha,nombre_producto,precio_unitario,total_venta
0,1,101,2.0,2026-01-05,Notebook,850000,1700000.0
1,2,102,1.0,2026-01-06,Monitor,310000,310000.0
2,3,103,3.0,2026-01-07,Teclado,52000,156000.0
3,4,104,2.0,2026-01-08,Mouse,27000,54000.0
4,5,105,4.0,2026-01-09,Auriculares,78000,312000.0


# Paso 9: Validación del resultado

Se verifica que no existan valores nulos en las columnas críticas del análisis.

In [12]:
columnas_criticas = [
    "id_producto",
    "cantidad",
    "fecha",
    "nombre_producto",
    "precio_unitario",
    "total_venta"
]

ventas[columnas_criticas].isnull().sum()

,0
id_producto,0
cantidad,0
fecha,0
nombre_producto,0
precio_unitario,0
total_venta,0


# Paso 10: Resumen de calidad de datos

Se realiza una validación final para confirmar que el dataset está listo para su exportación.

In [13]:
print("Cantidad de registros:", len(ventas))
print("Nulos totales:", ventas.isnull().sum().sum())
print("Monto total vendido:", ventas["total_venta"].sum())

assert ventas.isnull().sum().sum() == 0

print("Proyecto validado correctamente ✅")

Cantidad de registros: 9
Nulos totales: 0
Monto total vendido: 4215000.0
Proyecto validado correctamente ✅


# Paso 11: Exportación a Parquet

Finalmente se exporta el dataset integrado a formato Parquet.

Este formato permite almacenar información de manera más eficiente que un archivo CSV tradicional.

In [14]:
ventas.to_parquet(
    f"{ruta}/ventas_integradas.parquet",
    index=False,
    engine="pyarrow"
)

print("Archivo Parquet generado correctamente.")

Archivo Parquet generado correctamente.


In [15]:
import os

archivo_generado = os.path.exists(
    f"{ruta}/ventas_integradas.parquet"
)

print("¿Existe el archivo?", archivo_generado)

¿Existe el archivo? True


# Conclusión

En esta pre-entrega se desarrolló un proceso completo de integración y transformación de datos a partir de dos fuentes con formatos diferentes: un archivo CSV con las transacciones de venta y un archivo Excel con el maestro de productos.

Antes de integrar la información, fue necesario revisar la estructura y los tipos de datos de ambas fuentes. En las transacciones se detectaron valores faltantes en `cantidad` y `fecha`, por lo que se aplicaron criterios diferentes según el tipo de información: la cantidad faltante se completó mediante la mediana, mientras que el registro sin fecha fue eliminado debido a que no permitía identificar correctamente el momento de la operación. También se convirtió `fecha` a formato datetime y se verificó la compatibilidad de `id_producto` entre ambos datasets.

Una vez preparados los datos, se realizó el merge mediante `id_producto`, incorporando a cada transacción el nombre y el precio unitario correspondiente. A partir de esta integración se generó la nueva variable `total_venta`, calculada como la cantidad vendida multiplicada por el precio unitario.

Como resultado, se obtuvo un dataset final de **9 registros, sin valores nulos en los campos críticos y con un monto total de ventas de $4.215.000**. Finalmente, la información fue exportada correctamente a formato Parquet y se verificó la existencia del archivo generado.

De esta manera, el trabajo no solo permitió integrar dos fuentes diferentes, sino también aplicar controles de calidad antes y después de la integración. El resultado es un dataset limpio, consistente y preparado para ser utilizado en futuros análisis, mostrando de forma práctica las principales etapas de un proceso ETL: extracción, transformación e integración de los datos y generación de una salida optimizada.

In [16]:
import pandas as pd
import openpyxl
import pyarrow

print("pandas:", pd.__version__)
print("openpyxl:", openpyxl.__version__)
print("pyarrow:", pyarrow.__version__)

pandas: 2.2.3
openpyxl: 3.1.5
pyarrow: 23.0.1
